In [19]:
from google.colab import drive
drive.mount('/content/drive')

dir_ = '/content/drive/MyDrive/Cohand/Minh học data/Kỳ 3/NLP/'
raw_data = dir_ + 'data/raw/'
processed_data = dir_ + 'data/processed/'
requirment_file = dir_ + 'requirements.txt'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
packages = ['underthesea', 'pyvi']
package_string = " ".join(packages)

!pip install {package_string} -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.0 MB/s eta 0:00:00


In [27]:
import pandas as pd
import numpy as np
import re
from underthesea import word_tokenize
from pyvi import ViTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer



In [24]:
# load data
df = pd.read_csv(raw_data + 'full_data.csv')
df = df.dropna(subset=['data'])
print(df.shape)
df.head(5)

(16227, 9)


,data,stayingpower,texture,smell,price,others,colour,shipping,packing
0,Công dụng: tốt\r\nKết cấu: đẹp\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN
1,Công dụng: son môi\r\nKết cấu: khô\r\nĐộ bền m...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN
2,"Son mịn, mùi thơm nhẹ, lâu trôi.\r\nVideo+ hìn...",positive,positive,positive,NaN,NaN,NaN,NaN,NaN
3,Công dụng: đánh son\r\nKết cấu: Đóng gói cẩn t...,positive,NaN,NaN,positive,NaN,NaN,negative,NaN
4,Công dụng: tốt\r\nKết cấu: tốt\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,neutral,NaN,NaN


# Làm sạch dữ liệu

In [35]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)

    words = text.split()
    filtered_words = []

    for word in words:
        if len(word) > 20:
            continue

        # 'ngonnnnn' -> 'ngon'
        word = re.sub(r'([a-z])\1{2,}', r'\1', word)

        # bỏ các kí tự f, z, w, j
        rare_chars = len(re.findall(r'[fzwj]', word))
        if len(word) > 5 and (rare_chars / len(word)) > 0.3:
            continue

        filtered_words.append(word)

    text = ' '.join(filtered_words)

    # Word Segmentation
    if text.strip():
        text = ViTokenizer.tokenize(text)

    return text


print("Cleaning text...")
df['processed_data'] = df['data'].apply(clean_text)

# Loại bỏ những dòng mà sau khi clean bị trống
df = df[df['processed_data'].str.strip() != '']
df = df[df['processed_data'].notna()]
df

Cleaning text...


,data,stayingpower,texture,smell,price,others,colour,shipping,packing,processed_data,labels
0,Công dụng: tốt\r\nKết cấu: đẹp\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN,công_dụng tốt kết_cấu đẹp độ bền màu lâu,"[stayingpower_positive, texture_positive]"
1,Công dụng: son môi\r\nKết cấu: khô\r\nĐộ bền m...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN,công_dụng son môi kết_cấu khô độ bền màu tuyệt...,"[stayingpower_positive, texture_positive]"
2,"Son mịn, mùi thơm nhẹ, lâu trôi.\r\nVideo+ hìn...",positive,positive,positive,NaN,NaN,NaN,NaN,NaN,son mịn mùi thơm nhẹ lâu trôi video hình_ảnh m...,"[stayingpower_positive, texture_positive, smel..."
3,Công dụng: đánh son\r\nKết cấu: Đóng gói cẩn t...,positive,NaN,NaN,positive,NaN,NaN,negative,NaN,công_dụng đánh son kết_cấu đóng_gói cẩn_thận đ...,"[stayingpower_positive, price_positive, shippi..."
4,Công dụng: tốt\r\nKết cấu: tốt\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,neutral,NaN,NaN,công_dụng tốt kết_cấu tốt độ bền màu tốt đóng_...,"[stayingpower_positive, texture_positive, colo..."
...,...,...,...,...,...,...,...,...,...,...,...
16222,Son đẹp xịn lắm luôn đóng gói rất cẩn thận ❤️❤...,NaN,NaN,NaN,NaN,NaN,positive,NaN,positive,son đẹp xịn lắm luôn đóng_gói rất cẩn_thận nên...,"[colour_positive, packing_positive]"
16223,Hàng đặt trong đợt dịch nên đợi lâu. Nma sản p...,NaN,NaN,positive,NaN,NaN,positive,negative,positive,hàng đặt trong đợt dịch nên đợi lâu nma sản_ph...,"[smell_positive, colour_positive, shipping_neg..."
16224,"Sản phẩm đẹp giao hàng hơi lâu , son black rou...",NaN,NaN,NaN,positive,NaN,NaN,negative,NaN,sản_phẩm đẹp giao hàng hơi lâu son black rouge...,"[price_positive, shipping_negative]"
16225,Phải nói là xinh cả trong lẫn ngoài luôn ấy ch...,NaN,NaN,NaN,NaN,NaN,positive,NaN,NaN,phải nói là xinh cả trong lẫn ngoài luôn ấy ch...,[colour_positive]


# Tạo nhãn

In [36]:
aspects = ['stayingpower', 'texture', 'smell', 'price', 'others', 'colour', 'shipping', 'packing']

def extract_labels(row):
    labels = []
    for aspect in aspects:
        if pd.notna(row[aspect]):
            labels.append(f"{aspect}_{row[aspect].strip().lower()}")
    return labels

df['labels'] = df.apply(extract_labels, axis=1)
df

,data,stayingpower,texture,smell,price,others,colour,shipping,packing,processed_data,labels
0,Công dụng: tốt\r\nKết cấu: đẹp\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN,công_dụng tốt kết_cấu đẹp độ bền màu lâu,"[stayingpower_positive, texture_positive]"
1,Công dụng: son môi\r\nKết cấu: khô\r\nĐộ bền m...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN,công_dụng son môi kết_cấu khô độ bền màu tuyệt...,"[stayingpower_positive, texture_positive]"
2,"Son mịn, mùi thơm nhẹ, lâu trôi.\r\nVideo+ hìn...",positive,positive,positive,NaN,NaN,NaN,NaN,NaN,son mịn mùi thơm nhẹ lâu trôi video hình_ảnh m...,"[stayingpower_positive, texture_positive, smel..."
3,Công dụng: đánh son\r\nKết cấu: Đóng gói cẩn t...,positive,NaN,NaN,positive,NaN,NaN,negative,NaN,công_dụng đánh son kết_cấu đóng_gói cẩn_thận đ...,"[stayingpower_positive, price_positive, shippi..."
4,Công dụng: tốt\r\nKết cấu: tốt\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,neutral,NaN,NaN,công_dụng tốt kết_cấu tốt độ bền màu tốt đóng_...,"[stayingpower_positive, texture_positive, colo..."
...,...,...,...,...,...,...,...,...,...,...,...
16222,Son đẹp xịn lắm luôn đóng gói rất cẩn thận ❤️❤...,NaN,NaN,NaN,NaN,NaN,positive,NaN,positive,son đẹp xịn lắm luôn đóng_gói rất cẩn_thận nên...,"[colour_positive, packing_positive]"
16223,Hàng đặt trong đợt dịch nên đợi lâu. Nma sản p...,NaN,NaN,positive,NaN,NaN,positive,negative,positive,hàng đặt trong đợt dịch nên đợi lâu nma sản_ph...,"[smell_positive, colour_positive, shipping_neg..."
16224,"Sản phẩm đẹp giao hàng hơi lâu , son black rou...",NaN,NaN,NaN,positive,NaN,NaN,negative,NaN,sản_phẩm đẹp giao hàng hơi lâu son black rouge...,"[price_positive, shipping_negative]"
16225,Phải nói là xinh cả trong lẫn ngoài luôn ấy ch...,NaN,NaN,NaN,NaN,NaN,positive,NaN,NaN,phải nói là xinh cả trong lẫn ngoài luôn ấy ch...,[colour_positive]


# Lưu preprocessed

In [37]:
df.to_csv(processed_data+'processed.csv', index=False)